In [6]:

import pandas as pd

data = {
    'Sky': ['Sunny', 'Sunny', 'Rainy', 'Sunny'],
    'AirTemp': ['Warm', 'Warm', 'Cold', 'Warm'],
    'Humidity': ['Normal', 'High', 'High', 'High'],
    'Wind': ['Strong', 'Strong', 'Strong', 'Strong'],
    'Water': ['Warm', 'Warm', 'Warm', 'Cool'],
    'Forecast': ['Same', 'Same', 'Change', 'Same'],
    'EnjoySport': ['Yes', 'Yes', 'No', 'Yes']
}

df = pd.DataFrame(data)
print("Training Dataset:")
print(df)

# Initialize the hypothesis
hypothesis = None

# Apply FIND-S algorithm
for _, row in df.iterrows():
    if row['EnjoySport'] == 'Yes':
        example = list(row[:-1])

        if hypothesis is None:
            hypothesis = example
        else:
            for i in range(len(hypothesis)):
                if hypothesis[i] != example[i]:
                    hypothesis[i] = '?'

print("\nFinal Hypothesis:")
print(hypothesis)


Training Dataset:
     Sky AirTemp Humidity    Wind Water Forecast EnjoySport
0  Sunny    Warm   Normal  Strong  Warm     Same        Yes
1  Sunny    Warm     High  Strong  Warm     Same        Yes
2  Rainy    Cold     High  Strong  Warm   Change         No
3  Sunny    Warm     High  Strong  Cool     Same        Yes

Final Hypothesis:
['Sunny', 'Warm', '?', 'Strong', '?', 'Same']


In [7]:
# Candidate-Elimination Algorithm
training_data = [
    (["Sunny", "Warm", "Normal", "Strong", "Warm", "Same"], "Yes"),
    (["Sunny", "Warm", "High", "Strong", "Warm", "Same"], "Yes"),
    (["Rainy", "Cold", "High", "Strong", "Warm", "Change"], "No"),
    (["Sunny", "Warm", "High", "Strong", "Cool", "Change"], "Yes"),
]

num_attributes = len(training_data[0][0])
domains = [sorted({x[i] for x, _ in training_data}) for i in range(num_attributes)]
S = ["Ø"] * num_attributes
G = [["?"] * num_attributes]

def covers(hypothesis, example):
    return all(h == "?" or h == value for h, value in zip(hypothesis, example))

def more_general_or_equal(h1, h2):

    return all(a == "?" or a == b for a, b in zip(h1, h2))

for example, label in training_data:
    if label == "Yes":

        G = [g for g in G if covers(g, example)]

        for i, value in enumerate(example):
            if S[i] == "Ø":
                S[i] = value
            elif S[i] != value:
                S[i] = "?"

        G = [g for g in G if more_general_or_equal(g, S)]
    else:

        new_G = []
        for g in G:
            if covers(g, example):
                for i in range(num_attributes):
                    if g[i] == "?" and S[i] not in ("?", "Ø") and S[i] != example[i]:
                        h = g.copy()
                        h[i] = S[i]
                        if covers(h, example) is False and more_general_or_equal(h, S):
                            if h not in new_G:
                                new_G.append(h)
            else:
                new_G.append(g)
        G = new_G
    G = [g for g in G if not any(g != other and more_general_or_equal(g, other)
                                  for other in G)]

print("Specific Boundary (S):")
print(S)
print("General Boundary (G):")
for hypothesis in G:
    print(hypothesis)

Specific Boundary (S):
['Sunny', 'Warm', '?', 'Strong', '?', '?']
General Boundary (G):
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']
